# 🧪 QTrader v4.7: Institutional Strategy Lab

Welcome to the **Strategy Lab**. This notebook is designed for A/B testing trading strategies side-by-side using institutional quant metrics (Sortino, Calmar, Confidence Intervals).

**Objective**: Compare our old trend-following model (Baseline) against a new Multi-Factor Alpha Confluence model (MACD + RSI + VPIN + Trend) on 5-minute data.

In [16]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from qtrader.core.config import Config
from qtrader.core.event import OrderEvent
from qtrader.input.data.market.coinbase_market import CoinbaseMarketDataClient
from qtrader.output.execution.paper_engine import PaperTradingEngine
from qtrader.output.analytics.ev_calculator import EVCalculator

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_dark"


## 1. Global Configuration & Market Data

In [10]:
SYMBOL = "BTC-USD"
CAPITAL = 10000.0
FEE_RATE = 0.0004  # 0.04% Tier
GRANULARITY = "FIVE_MINUTE"
LOOKBACK_DAYS = 360

client = CoinbaseMarketDataClient()
end_dt = datetime.now(Config.tz)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)

candles = client.get_candles(SYMBOL, GRANULARITY, start=start_dt, end=end_dt)
df = candles.to_pandas()
print(f"Fetched {len(df)} candles for {SYMBOL} from {start_dt.date()} to {end_dt.date()}")


Fetched 103609 candles for BTC-USD from 2025-03-21 to 2026-03-16


## 2. Baseline Strategy: EMA Crossover
The old strategy that suffered from 'Penny-Scale' wins and lagging false breakouts.

In [11]:
def run_baseline_strategy(df: pd.DataFrame) -> tuple:
    df_base = df.copy()
    df_base['ema_fast'] = df_base['close'].ewm(span=50, adjust=False).mean()
    df_base['ema_slow'] = df_base['close'].ewm(span=200, adjust=False).mean()
    
    engine = PaperTradingEngine(starting_capital=CAPITAL, fee_rate=FEE_RATE)
    holding = False
    
    for i in range(200, len(df_base)-1):
        row = df_base.iloc[i]
        prev_row = df_base.iloc[i-1]
        # FIX: Entry occurs at NEXT OPEN to avoid look-ahead bias
        next_open = float(df_base.iloc[i+1]["open"])
        
        market_state = {
            "bid": next_open,
            "ask": next_open,
            "top_depth": 50.0,
            "venue": "Coinbase_ADV"
        }

        signal_buy = (prev_row['ema_fast'] <= prev_row['ema_slow']) and (row['ema_fast'] > row['ema_slow'])
        signal_sell = (prev_row['ema_fast'] >= prev_row['ema_slow']) and (row['ema_fast'] < row['ema_slow'])

        if not holding and signal_buy:
            qty = round((CAPITAL * 0.10) / next_open, 4)
            entry_event = OrderEvent(
                symbol=SYMBOL, order_type="MARKET", side="BUY", 
                quantity=qty, price=next_open
            )
            engine.simulate_fill(entry_event, market_state)
            holding = True
        elif holding and signal_sell:
            exit_event = OrderEvent(
                symbol=SYMBOL, order_type="MARKET", side="SELL", 
                quantity=qty, price=next_open
            )
            engine.simulate_fill(exit_event, market_state)
            holding = False

    calc = EVCalculator(engine.closed_trades, fee_rate=FEE_RATE)
    return calc.diagnose(SYMBOL, min_trades=300), engine.closed_trades

baseline_report, baseline_trades = run_baseline_strategy(df)
print(f"Baseline EV per trade: {baseline_report.ev_per_trade:.6f}")


Baseline EV per trade: -0.241239


## 3. Advanced Strategy: Multi-Factor Confluence
Requires 3 independent vectors of agreement: MACD momentum, RSI recovery from extreme dip, and Trend alignment. We also use a tight `1x ATR` stop and `3x ATR` target.

In [12]:
def run_confluence_strategy(df: pd.DataFrame) -> tuple:
    df_new = df.copy()
    
    # 1. Trend Anchor
    df_new['ema_200'] = df_new['close'].ewm(span=200, adjust=False).mean()
    
    # 2. MACD (12, 26, 9)
    ema_12 = df_new['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df_new['close'].ewm(span=26, adjust=False).mean()
    df_new['macd'] = ema_12 - ema_26
    df_new['macd_signal'] = df_new['macd'].ewm(span=9, adjust=False).mean()
    df_new['macd_hist'] = df_new['macd'] - df_new['macd_signal']
    
    # 3. RSI (14)
    delta = df_new['close'].diff()
    gain = delta.clip(lower=0)
    loss = -1 * delta.clip(upper=0)
    avg_gain = gain.ewm(com=13, adjust=False).mean()
    avg_loss = loss.ewm(com=13, adjust=False).mean()
    rs = avg_gain / avg_loss
    df_new['rsi'] = 100 - (100 / (1 + rs))
    
    # 4. Volatility (ATR 14)
    high_low = df_new['high'] - df_new['low']
    high_close = np.abs(df_new['high'] - df_new['close'].shift())
    low_close = np.abs(df_new['low'] - df_new['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    df_new['atr'] = true_range.rolling(14).mean()
    
    engine = PaperTradingEngine(starting_capital=CAPITAL, fee_rate=FEE_RATE)
    holding = False
    entry_price = 0.0
    sl_price = 0.0
    tp_price = 0.0
    bars_held = 0
    
    for i in range(200, len(df_new)-1):
        row = df_new.iloc[i]
        prev_row = df_new.iloc[i-1]
        next_open = float(df_new.iloc[i+1]["open"])
        next_low = float(df_new.iloc[i+1]["low"])
        next_high = float(df_new.iloc[i+1]["high"])
        
        market_state = {"bid": next_open, "ask": next_open, "top_depth": 50.0, "venue": "Coinbase_ADV"}

        if not holding:
            # Votes
            trend_vote = row['close'] > row['ema_200']
            macd_vote = (prev_row['macd_hist'] < 0) and (row['macd_hist'] > 0)
            
            # RSI dip recovery: RSI was < 40 highly recently, but is now clawing back > 45
            rsi_recent_dip = df_new['rsi'].iloc[i-4:i].min() < 40
            rsi_vote = rsi_recent_dip and (row['rsi'] > 45) and (row['rsi'] < 60)
            
            # Volatility gate
            vol_vote = (row['atr'] / row['close']) > 0.0025  # Want at least 25bps ATR to cover 8bps fee
            
            # Entry logic
            if trend_vote and macd_vote and rsi_vote and vol_vote:
                qty = round((CAPITAL * 0.10) / next_open, 4)
                entry_event = OrderEvent(symbol=SYMBOL, order_type="MARKET", side="BUY", quantity=qty, price=next_open)
                engine.simulate_fill(entry_event, market_state)
                holding = True
                entry_price = next_open
                bars_held = 0
                
                # Asymmetric R:R (3:1)
                tp_price = entry_price + (3.0 * row['atr'])
                sl_price = entry_price - (1.0 * row['atr'])
                
        else:
            bars_held += 1
            exit_price = None
            reason = ""
            
            # Intra-bar stop check
            if next_low <= sl_price:
                # Slippage asymmetry: apply 1.5x penalty to slippage on panic sell
                exit_price = sl_price * 0.9995  
                reason = "STOP_LOSS"
            elif next_high >= tp_price:
                exit_price = tp_price
                reason = "TAKE_PROFIT"
            elif bars_held > 48: # Time stop (4 hours)
                exit_price = next_open
                reason = "TIME_STOP"
                
            if exit_price is not None:
                market_state["bid"] = exit_price
                exit_event = OrderEvent(symbol=SYMBOL, order_type="MARKET", side="SELL", quantity=qty, price=exit_price)
                engine.simulate_fill(exit_event, market_state)
                holding = False

    calc = EVCalculator(engine.closed_trades, fee_rate=FEE_RATE)
    return calc.diagnose(SYMBOL, min_trades=300), engine.closed_trades

confluence_report, confluence_trades = run_confluence_strategy(df)
print(f"Confluence EV per trade: {confluence_report.ev_per_trade:.6f}")


Confluence EV per trade: -0.787943


## 4. Institutional Assessment (A/B Comparison)

In [13]:
comparison = pd.DataFrame({
    "Metric": [
        "Total Trades", "Win Rate", "EV per Trade", "EV 95% CI", "Profit Factor", 
        "Sharpe Ratio (Corr.)", "Sortino Ratio", "Calmar Ratio", "Payoff Ratio",
        "Break-Even Win Rate", "Cost-to-Profit", "Status"
    ],
    "Baseline (EMA)": [
        baseline_report.total_trades,
        f"{baseline_report.win_rate:.2%}",
        f"{baseline_report.ev_per_trade:.4f}",
        f"± {baseline_report.ev_confidence_interval:.4f}",
        f"{baseline_report.profit_factor:.2f}",
        f"{baseline_report.sharpe_ratio:.2f}",
        f"{baseline_report.sortino_ratio:.2f}",
        f"{baseline_report.calmar_ratio:.2f}" if baseline_report.calmar_ratio != float("inf") else "INF",
        f"{baseline_report.payoff_ratio:.2f}",
        f"{baseline_report.break_even_win_rate:.2%}",
        f"{baseline_report.cost_to_profit_ratio:.2%}",
        baseline_report.status
    ],
    "Confluence": [
        confluence_report.total_trades,
        f"{confluence_report.win_rate:.2%}",
        f"{confluence_report.ev_per_trade:.4f}",
        f"± {confluence_report.ev_confidence_interval:.4f}",
        f"{confluence_report.profit_factor:.2f}",
        f"{confluence_report.sharpe_ratio:.2f}",
        f"{confluence_report.sortino_ratio:.2f}",
        f"{confluence_report.calmar_ratio:.2f}" if confluence_report.calmar_ratio != float("inf") else "INF",
        f"{confluence_report.payoff_ratio:.2f}",
        f"{confluence_report.break_even_win_rate:.2%}",
        f"{confluence_report.cost_to_profit_ratio:.2%}",
        confluence_report.status
    ]
})

from IPython.display import display
display(comparison)


,Metric,Baseline (EMA),Confluence
0,Total Trades,314,40
1,Win Rate,25.48%,30.00%
2,EV per Trade,-0.2412,-0.7879
3,EV 95% CI,± 1.7121,± 1.8806
4,Profit Factor,0.95,0.76
5,Sharpe Ratio (Corr.),-2.22,-18.26
6,Sortino Ratio,-4.82,-28.30
7,Calmar Ratio,-17.18,-245.33
8,Payoff Ratio,2.79,1.76
9,Break-Even Win Rate,26.42%,36.17%


In [14]:
# Equity Curve Comparison
pnl_base = np.cumsum([t.pnl for t in baseline_trades]) if baseline_trades else [0]
pnl_conf = np.cumsum([t.pnl for t in confluence_trades]) if confluence_trades else [0]

fig = go.Figure()
fig.add_trace(go.Scatter(y=pnl_base, name="Baseline (EMA)", mode="lines"))
fig.add_trace(go.Scatter(y=pnl_conf, name="RSI-MACD Confluence", mode="lines"))
fig.update_layout(title="Cumulative PnL Comparison", xaxis_title="Trade Number", yaxis_title="Net Profit (USD)")
fig.show()


## 5. Daily Regime & Auto Selector

In [15]:
# Analyzes the last 24 hours to determine the current regime 
# and programmatically select the best sub-strategy to deploy.
last_24h = df.iloc[-288:] # 288 5-min candles in 24h
atr_24h = last_24h['high'].max() - last_24h['low'].min()
trend_24h = (last_24h['close'].iloc[-1] / last_24h['close'].iloc[0]) - 1.0

regime = "SIDEWAYS"
if trend_24h > 0.02:
    regime = "TRENDING_UP"
elif trend_24h < -0.02:
    regime = "TRENDING_DOWN"

print(f"Current Market Regime (Last 24h): {regime}")
print(f"24h Range: {atr_24h:.2f} USD")

if regime in ["TRENDING_UP", "TRENDING_DOWN"]:
    print("🤖 Auto-Selector: Deploying MACD_Confluence (Trend is your friend)")
else:
    print("🤖 Auto-Selector: Deploying Mean_Reversion (Range-bound market)")


Current Market Regime (Last 24h): TRENDING_UP
24h Range: 3169.99 USD
🤖 Auto-Selector: Deploying MACD_Confluence (Trend is your friend)
